# CropRecommender: LLM-as-an-Agronomist via ClusterFewshot

This notebook demonstrates how to use **ClusterFewshot** to derive quality few-shot demonstrations for a crop recommendation system. Given soil nutrient levels and environmental conditions, the LLM recommends the optimal crop with expert agronomic reasoning.

**Dataset:** [Kaggle Crop Recommendation Dataset](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset/data) — 2200 rows, 22 crops, 7 numeric features.

## 1. Setup & Imports

In [ ]:
import os
import dspy
import numpy as np

from dspy.evaluate import Evaluate
from programs import CropRecommender
from dspy.datasets.crop_recommendation import (
    CropRecommendationDataset,
    crop_recommendation_metric,
    create_crop_numeric_encoder,
    format_observation,
    CROP_NUMERIC_FIELDS,
)
from dspy.teleprompt.clusterfewshot import ClusterFewshot

dspy.settings.experimental = True

## 2. Load the Dataset

Download the CSV from [Kaggle](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset/data) and set the path below.

In [ ]:
CSV_PATH = "/path/to/Crop_recommendation.csv"  # <-- Update this

dataset = CropRecommendationDataset(csv_path=CSV_PATH)
trainset, devset, testset = dataset.get_data_splits()

print(f"Train: {len(trainset)}, Dev: {len(devset)}, Test: {len(testset)}")

## 3. Inspect the Example Schema

Each example carries a **textual observation** (LLM input), **7 numeric fields** (encoder input), and a **crop label** (output).

In [ ]:
ex = trainset[0]

print("=== Full Example ===")
for k, v in ex.items():
    print(f"  {k}: {v}")

print(f"\n=== Inputs (what the LLM sees) ===")
for k, v in ex.inputs().items():
    print(f"  {k}: {v}")

print(f"\n=== Labels (includes crop + auxiliary numeric fields) ===")
for k, v in ex.labels().items():
    print(f"  {k}: {v}")

## 4. The CropNumericEncoder

The custom encoder extracts 7 physical measurements as embedding vectors for clustering. This lets ClusterFewshot discover **latent agronomic zones** — groups of field conditions with similar growing requirements.

In [ ]:
encoder = create_crop_numeric_encoder()
print(f"Encoder: {encoder}")
print(f"Fields used: {CROP_NUMERIC_FIELDS}")

# Encode a few examples
sample = trainset[:5]
embeddings = encoder.encode(sample)
print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Example vector: {embeddings[0]}")
print(f"Corresponding crop: {sample[0].crop}")

## 5. The CropRecommender Program

A `ChainOfThought` module that takes an observation and produces reasoning + crop recommendation.

In [ ]:
student = CropRecommender()

# Inspect the signature
for name, predictor in student.named_predictors():
    print(f"Predictor: {name}")
    print(f"Signature: {predictor.signature}")

## 6. Configure LM

In [ ]:
MODEL = "gemini/gemini-2.0-flash"  # <-- Update if needed

lm = dspy.LM(MODEL, api_key=os.getenv("GEMINI_API_KEY"))
dspy.configure(lm=lm)

## 7. Compile with ClusterFewshot

ClusterFewshot will:
1. **Bootstrap** — run the student on training examples, keep those that pass the metric
2. **Cluster** — use the CropNumericEncoder to group examples into agronomic zones
3. **Rank** — evaluate each example as a one-shot demo on a validation subset
4. **Select** — pick the best demo from each cluster (`best_in_cluster` for classification)

In [ ]:
optimizer = ClusterFewshot(
    metric=crop_recommendation_metric,
    task_type="classification",
    semantic_encoders=[create_crop_numeric_encoder()],
    apply_visuals=True,
)

optimized_program = optimizer.compile(
    student=student,
    trainset=trainset,
    valset=devset,
)

## 8. Inspect Selected Demonstrations

These are the few-shot examples ClusterFewshot selected — one per agronomic zone.

In [ ]:
for name, predictor in optimized_program.named_predictors():
    print(f"Predictor '{name}': {len(predictor.demos)} demonstrations\n")
    for i, demo in enumerate(predictor.demos):
        print(f"  Demo {i+1}:")
        for k, v in demo.items():
            val_str = str(v)[:120] + "..." if len(str(v)) > 120 else str(v)
            print(f"    {k}: {val_str}")
        print()

## 9. Evaluate on Test Set

In [ ]:
evaluator = Evaluate(
    devset=testset,
    metric=crop_recommendation_metric,
    num_threads=8,
    display_progress=True,
    display_table=False,
)

accuracy = evaluator(optimized_program)
print(f"\nTest accuracy: {accuracy:.2f}%")

## 10. Interactive Inference

Try the optimized agronomist with custom observations.

In [ ]:
# Example: High-rainfall tropical conditions
observation = (
    "Soil Analysis — Nitrogen (N): 85 mg/kg, Phosphorous (P): 58 mg/kg, "
    "Potassium (K): 41 mg/kg. Environmental Conditions — Temperature: 23.50°C, "
    "Humidity: 80.00%, Soil pH: 6.80, Rainfall: 220.00 mm."
)

result = optimized_program(observation=observation)
print(f"Observation: {observation}")
print(f"\nRecommended crop: {result.crop}")
print(f"Reasoning: {result.reasoning}")

In [ ]:
# Example: Arid, high-temperature conditions
observation = (
    "Soil Analysis — Nitrogen (N): 25 mg/kg, Phosphorous (P): 65 mg/kg, "
    "Potassium (K): 20 mg/kg. Environmental Conditions — Temperature: 34.00°C, "
    "Humidity: 40.00%, Soil pH: 7.80, Rainfall: 50.00 mm."
)

result = optimized_program(observation=observation)
print(f"Observation: {observation}")
print(f"\nRecommended crop: {result.crop}")
print(f"Reasoning: {result.reasoning}")

In [ ]:
# Example: Fruit-growing conditions (high potassium)
observation = (
    "Soil Analysis — Nitrogen (N): 30 mg/kg, Phosphorous (P): 120 mg/kg, "
    "Potassium (K): 200 mg/kg. Environmental Conditions — Temperature: 28.00°C, "
    "Humidity: 65.00%, Soil pH: 6.20, Rainfall: 110.00 mm."
)

result = optimized_program(observation=observation)
print(f"Observation: {observation}")
print(f"\nRecommended crop: {result.crop}")
print(f"Reasoning: {result.reasoning}")